In [1]:
from environment import Environment
from nnsight import LanguageModel
import sys
import torch as t
from getpass import getpass
import os

from keys import REPLICATE
os.environ["REPLICATE_API_TOKEN"] = REPLICATE

# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [2]:
model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [3]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

local_dir = "../jbloom_dictionaries"

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir=local_dir)

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [4]:
os.environ["PROVIDER"] = "replicate"

In [5]:
from llama_config import EnvConfig, ExplainerConfig, CondenserConfig, DetectionScorerConfig, GenerationScorerConfig

env_config = EnvConfig()
explainer_cfg = ExplainerConfig()
condenser_cfg = CondenserConfig()
d_scorer_cfg = DetectionScorerConfig()
gen_scorer_cfg = GenerationScorerConfig()

env = Environment(
    model=model, 
    sae=sae_list[0],
    cfg=env_config,
    provider="replicate"
)

In [6]:
import threading
layer = 0

def feature_thread(environment, layer, feature_id, explainer_cfg, condenser_cfg, d_scorer_cfg, gen_scorer_cfg):
    environment.execute(layer, feature_id, explainer_cfg, condenser_cfg, d_scorer_cfg, gen_scorer_cfg)

# Assuming initialization of Environment, config, and other necessary components here
env.load_cache(layer)

# Example of running features in separate threads
threads = []

for feature_id in [7000]:  # Assuming you want to process 10 features
    thread = threading.Thread(target=feature_thread, args=(env, layer, feature_id, explainer_cfg, condenser_cfg, d_scorer_cfg, gen_scorer_cfg))
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

100%|██████████| 13/13 [00:36<00:00,  2.84s/it]


Activation Cache Size: torch.Size([1950, 128, 24576])


Processing batches: 100%|██████████| 2/2 [00:13<00:00,  6.71s/it]


Explainer completed.


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

100%|██████████| 2/2 [00:04<00:00,  2.23s/it]


Detection Scorer completed.


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Generation Scorer completed.


┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Content                                                                                                  ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Section   │ Running explainer.                                                                                       │
├───────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ User      │  You are a meticulous AI researcher conducting an important investigation into a certain neuron in a     │
│           │ language model.                                                                                          │
│           │                                                                                                          │
│           │ You will be given a list of text examples on which the neuron activates. The specific tokens which cause │
│           │ the neuron to activate will appear between delimiters like <<this>>. If a sequence of consecutive tokens │
│           │ all cause the neuron to activate, the entire sequence of tokens will be contained between delimiters     │
│           │ <<just like this>>.                                                                                      │
│           │                                                                                                          │
│           │ Your task is to understand what features of the input text cause the neuron to activate.                 │
│           │                                                                                                          │
│           │ You must follow these steps:                                                                             │
│           │                                                                                                          │
│           │ Step 1: For each text example in turn, note which tokens (i.e. words, fragments of words, or symbols)    │
│           │ caused the neuron to activate. Then note which tokens came before the activating tokens. Then note which │
│           │ tokens came after the activating tokens.                                                                 │
│           │ Step 2: Look for patterns in the tokens you noted down in Step 1.                                        │
│           │ Step 3: Write down several general shared features of the text examples.                                 │
│           │ Step 4: Look at the shared features you found, as well as patterns in the tokens you wrote down in Steps │
│           │ 1 and 2, to produce 3 explanations for what features of text cause the neuron to activate. The final 2   │
│           │ lines of your output must consist of your 2 explanations.                                                │
│           │                                                                                                          │
│           │ Guidelines:                                                                                              │
│           │                                                                                                          │
│           │ - Avoid using words like "often", "particularly" or "especially" in your explanations. Either a feature  │
│           │ is relevant to the explanation, or it isn't. There is no in between.                                     │
│           │ - Think very carefully and show your work.                                                               │
│           │                                                                                                          │
│           │ Get ready to see the list of text examples.                                                              │
│      